# Análise RFM — Segmentação de Clientes | Olist E-commerce

**Objetivo:** Segmentar os clientes da Olist com base em comportamento de compra usando a metodologia RFM,
identificando quais segmentos geram mais valor e quais estão em risco.

**Dataset:** Olist Brazilian E-commerce (Kaggle)  
**Ferramentas:** Python · Pandas · Matplotlib · Seaborn · SQLite

---

## O que é RFM?

| Dimensão | Descrição | Pontuação |
|----------|-----------|-----------|
| **R — Recência** | Quantos dias desde a última compra? | 5 = comprou recentemente |
| **F — Frequência** | Quantas vezes o cliente comprou? | 5 = compra muito |
| **M — Valor Monetário** | Quanto o cliente gastou no total? | 5 = gasto alto |

Cada cliente recebe uma nota de 1 a 5 em cada dimensão.  
A combinação das três notas define o **segmento** do cliente.


## 1. Importação de Bibliotecas

In [ ]:
import sqlite3
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
import warnings

warnings.filterwarnings('ignore')

# Configurações de visualização
plt.rcParams.update({
    'figure.dpi': 130,
    'font.family': 'sans-serif',
    'axes.spines.top': False,
    'axes.spines.right': False,
})
sns.set_palette('muted')

print("Bibliotecas carregadas com sucesso!")

## 2. Conexão ao Banco de Dados e Extração dos Dados

In [ ]:
# ⚠️  Atualize o caminho abaixo para o local do seu arquivo SQLite
DB_PATH = 'db_olist.sqlite'

conn = sqlite3.connect(DB_PATH)

query = """
    SELECT
        c.customer_unique_id,
        o.order_id,
        o.order_purchase_timestamp,
        op.payment_value
    FROM orders o
    JOIN customer c        ON o.customer_id  = c.customer_id
    JOIN order_payments op ON o.order_id     = op.order_id
    WHERE o.order_status = 'delivered'
      AND op.payment_value > 0
"""

df_raw = pd.read_sql(query, conn, parse_dates=['order_purchase_timestamp'])
conn.close()

print(f"Registros carregados: {len(df_raw):,}")
print(f"Clientes únicos:      {df_raw['customer_unique_id'].nunique():,}")
print(f"\nPrimeira compra:  {df_raw['order_purchase_timestamp'].min().date()}")
print(f"Última compra:    {df_raw['order_purchase_timestamp'].max().date()}")
df_raw.head()

## 3. Cálculo de R, F, M por Cliente

In [ ]:
# Data de referência: dia seguinte à última compra no dataset
# (simula o ponto de vista do analista na data do corte)
DATA_REF = df_raw['order_purchase_timestamp'].max() + pd.Timedelta(days=1)
print(f"Data de referência: {DATA_REF.date()}")

rfm = (
    df_raw
    .groupby('customer_unique_id')
    .agg(
        ultima_compra  = ('order_purchase_timestamp', 'max'),
        frequencia     = ('order_id',                 'nunique'),
        valor_total    = ('payment_value',             'sum')
    )
    .reset_index()
)

# R = dias desde a última compra (quanto menor, mais recente)
rfm['recencia_dias'] = (DATA_REF - rfm['ultima_compra']).dt.days

print(f"\nClientes na base RFM: {len(rfm):,}")
print("\nEstatísticas descritivas:")
rfm[['recencia_dias','frequencia','valor_total']].describe().round(2)

## 4. Scoring RFM (1 a 5 por quintis)

In [ ]:
# Recência: invertida — quem comprou mais recentemente recebe nota 5
rfm['R'] = pd.qcut(rfm['recencia_dias'], q=5,
                    labels=[5, 4, 3, 2, 1],   # 5 = mais recente
                    duplicates='drop').astype(int)

# Frequência: quem compra mais recebe nota 5
rfm['F'] = pd.qcut(rfm['frequencia'], q=5,
                    labels=[1, 2, 3, 4, 5],
                    duplicates='drop').astype(int)

# Valor monetário: quem gasta mais recebe nota 5
rfm['M'] = pd.qcut(rfm['valor_total'], q=5,
                    labels=[1, 2, 3, 4, 5],
                    duplicates='drop').astype(int)

# Score composto (para referência rápida)
rfm['rfm_score'] = rfm['R'].astype(str) + rfm['F'].astype(str) + rfm['M'].astype(str)

print("Scores calculados. Amostra:")
rfm[['customer_unique_id','recencia_dias','frequencia','valor_total','R','F','M','rfm_score']].head(10)

## 5. Segmentação de Clientes

In [ ]:
def classificar_segmento(row):
    r, f, m = row['R'], row['F'], row['M']

    if r >= 4 and f >= 4 and m >= 4:
        return 'Campeão'
    elif f >= 4 and m >= 3:
        return 'Cliente Fiel'
    elif r >= 4 and f >= 2:
        return 'Potencial Fiel'
    elif r >= 4 and f == 1:
        return 'Novo Cliente'
    elif r == 3:
        return 'Precisa Atenção'
    elif r <= 2 and f >= 3:
        return 'Em Risco'
    elif r == 2:
        return 'Hibernando'
    else:
        return 'Perdido'

rfm['segmento'] = rfm.apply(classificar_segmento, axis=1)

# Resumo por segmento
resumo = (
    rfm.groupby('segmento')
    .agg(
        clientes       = ('customer_unique_id', 'count'),
        receita_total  = ('valor_total',         'sum'),
        recencia_media = ('recencia_dias',        'mean'),
        freq_media     = ('frequencia',           'mean'),
        ticket_medio   = ('valor_total',          'mean')
    )
    .round(2)
    .sort_values('receita_total', ascending=False)
    .reset_index()
)

resumo['pct_clientes'] = (resumo['clientes'] / resumo['clientes'].sum() * 100).round(1)
resumo['pct_receita']  = (resumo['receita_total'] / resumo['receita_total'].sum() * 100).round(1)
print(resumo.to_string(index=False))

## 6. Visualizações

In [ ]:
# Paleta de cores por segmento
cores = {
    'Campeão':        '#1D9E75',
    'Cliente Fiel':   '#185FA5',
    'Potencial Fiel': '#3BAD6E',
    'Novo Cliente':   '#7F77DD',
    'Precisa Atenção':'#BA7517',
    'Em Risco':       '#D85A30',
    'Hibernando':     '#D4537E',
    'Perdido':        '#A32D2D',
}

ordem = list(cores.keys())
fig, axes = plt.subplots(1, 2, figsize=(16, 6))
fig.suptitle('Segmentação RFM — Olist E-commerce', fontsize=14, fontweight='bold', y=1.02)

# --- Gráfico 1: Clientes por segmento ---
seg_clientes = resumo.set_index('segmento').reindex(ordem)['clientes'].dropna()
bars1 = axes[0].barh(
    seg_clientes.index,
    seg_clientes.values,
    color=[cores[s] for s in seg_clientes.index],
    edgecolor='white', linewidth=0.5
)
axes[0].set_title('Número de Clientes por Segmento', fontweight='bold', pad=12)
axes[0].set_xlabel('Clientes')
axes[0].invert_yaxis()
for bar in bars1:
    w = bar.get_width()
    axes[0].text(w + 200, bar.get_y() + bar.get_height()/2,
                 f'{int(w):,}', va='center', ha='left', fontsize=9)

# --- Gráfico 2: Receita por segmento ---
seg_receita = resumo.set_index('segmento').reindex(ordem)['receita_total'].dropna()
bars2 = axes[1].barh(
    seg_receita.index,
    seg_receita.values / 1e6,
    color=[cores[s] for s in seg_receita.index],
    edgecolor='white', linewidth=0.5
)
axes[1].set_title('Receita Total por Segmento (R$ milhões)', fontweight='bold', pad=12)
axes[1].set_xlabel('Receita (R$ mi)')
axes[1].invert_yaxis()
for bar in bars2:
    w = bar.get_width()
    axes[1].text(w + 0.02, bar.get_y() + bar.get_height()/2,
                 f'R$ {w:.1f}M', va='center', ha='left', fontsize=9)

plt.tight_layout()
plt.savefig('rfm_segmentos.png', bbox_inches='tight', dpi=150)
plt.show()
print("Gráfico salvo: rfm_segmentos.png")

In [ ]:
# --- Scatter: Recência vs Frequência, colorido por segmento ---
fig, ax = plt.subplots(figsize=(12, 7))

for seg in ordem:
    subset = rfm[rfm['segmento'] == seg]
    ax.scatter(
        subset['recencia_dias'],
        subset['frequencia'],
        c=cores[seg], label=seg, alpha=0.5, s=20, edgecolors='none'
    )

ax.set_xlabel('Recência (dias desde última compra) ← mais recente', fontsize=11)
ax.set_ylabel('Frequência (nº de pedidos)', fontsize=11)
ax.set_title('RFM — Recência vs Frequência por Segmento', fontsize=13, fontweight='bold')
ax.invert_xaxis()  # clientes mais recentes aparecem à direita
ax.legend(title='Segmento', bbox_to_anchor=(1.01, 1), loc='upper left', fontsize=9)
plt.tight_layout()
plt.savefig('rfm_scatter.png', bbox_inches='tight', dpi=150)
plt.show()
print("Gráfico salvo: rfm_scatter.png")

## 7. Insights e Conclusões

In [ ]:
print("=" * 60)
print("RESUMO EXECUTIVO — ANÁLISE RFM")
print("=" * 60)

total_clientes = rfm['customer_unique_id'].nunique()
total_receita  = rfm['valor_total'].sum()

# Clientes de alto valor (Campeão + Fiel)
alto_valor = rfm[rfm['segmento'].isin(['Campeão', 'Cliente Fiel'])]
print(f"\nAlto Valor (Campeão + Fiel):")
print(f"  Clientes:  {len(alto_valor):,} ({len(alto_valor)/total_clientes*100:.1f}% da base)")
print(f"  Receita:   R$ {alto_valor['valor_total'].sum():,.0f} ({alto_valor['valor_total'].sum()/total_receita*100:.1f}% do total)")

# Clientes em risco de churn
em_risco = rfm[rfm['segmento'].isin(['Em Risco', 'Hibernando'])]
print(f"\nEm Risco de Churn (Em Risco + Hibernando):")
print(f"  Clientes:  {len(em_risco):,} ({len(em_risco)/total_clientes*100:.1f}% da base)")
print(f"  Receita:   R$ {em_risco['valor_total'].sum():,.0f} ({em_risco['valor_total'].sum()/total_receita*100:.1f}% do total)")

# Compra única (1-time buyers)
uma_compra = rfm[rfm['frequencia'] == 1]
print(f"\nCompradores únicos (1 pedido):")
print(f"  Clientes:  {len(uma_compra):,} ({len(uma_compra)/total_clientes*100:.1f}% da base)")
print(f"\n⚠️  Insight: A Olist é um marketplace — a maioria dos clientes compra apenas 1 vez.")
print("   Estratégias de retenção e cross-sell são essenciais para aumentar o LTV.")

print("\n" + "=" * 60)
print("Arquivo rfm_completo.csv gerado para uso no Power BI ou Excel.")
rfm.to_csv('rfm_completo.csv', index=False)
print("=" * 60)

---
## 📁 Arquivos Gerados

| Arquivo | Descrição |
|---------|-----------|
| `rfm_completo.csv` | Tabela com todos os clientes, scores e segmentos |
| `rfm_segmentos.png` | Gráfico de barras — clientes e receita por segmento |
| `rfm_scatter.png` | Scatter plot — recência vs frequência por segmento |

## 🎯 Próximos Passos Sugeridos

1. **Campeões** → Programa de fidelidade, early access a lançamentos
2. **Em Risco** → Campanha de reativação com desconto personalizado
3. **Novos Clientes** → Onboarding com e-mail de boas-vindas e segunda compra incentivada
4. **Perdidos** → Custo de reativação provavelmente alto — foque nos outros segmentos
